# Sentinel DDoS Mitigation - Cloud Trainer (Kaggle Edition)
This notebook trains the Sentinel DDoS binary classifier on modern high-performance hardware using Google Colab. It automatically downloads the two gold-standard datasets recommended for Sentinel:
- **CIC-IoT2023** (Modern Botnet/DDoS)
- **NF-UNSW-NB15-v2** (Subtle/Slow-rate threats)

### 🚀 Steps:
1. **Setup Environment**: Install dependencies and clone the Sentinel repo.
2. **Kaggle Ingestion**: Download datasets securely using Kaggle API.
3. **Train & Export**: Run `train_ml.py` to generate `ml_model.h`.
4. **Deploy**: Push the new model back to your repository.

## 1. Environment Setup

In [ ]:
# @title Install Dependencies
!pip install scikit-learn m2cgen numpy glob2 kaggle

import os

# @markdown --- 
# @markdown ### Git Configuration
REPO_URL = "https://github.com/navneetxdd/Sentinel-DDoS-Mitigation-System" # @param {type:"string"}
GITHUB_PAT = "" # @param {type:"string"}

if GITHUB_PAT:
    REPO_AUTH_URL = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
    !git clone {REPO_AUTH_URL} sentinel_repo
    %cd sentinel_repo/Sentinel_DDOS_Core
else:
    print("WARNING: No GITHUB_PAT provided. You won't be able to push changes.")
    !git clone {REPO_URL} sentinel_repo
    %cd sentinel_repo/Sentinel_DDOS_Core

## 2. Kaggle Dataset Ingestion
You need a Kaggle account. Go to Kaggle -> Settings -> Create New API Token, and paste the username/key below.

In [ ]:
# @title Download Kaggle Datasets
KAGGLE_USERNAME = "" # @param {type:"string"}
KAGGLE_KEY = "" # @param {type:"string"}

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

os.makedirs("data", exist_ok=True)

if KAGGLE_USERNAME and KAGGLE_KEY:
    print("Downloading CIC-IoT2023...")
    !kaggle datasets download -d fatihbilgin/cic-iot-2023 --unzip -p data/
    print("Downloading NF-UNSW-NB15-v2...")
    !kaggle datasets download -d dhoogla/nfunswnb15 --unzip -p data/
else:
    print("Skipping Kaggle download (No credentials provided).")

## 3. Train & Export


In [ ]:
# @title Automatically Patch `train_ml.py` for CIC-IoT2023 & NF-UNSW-NB15-v2
with open('train_ml.py', 'r', encoding='utf-8') as f:
    content = f.read()

target = "\"dns_query_count\": [\"ct_dns_query\"],\n}"
replacement = """\"dns_query_count\": [\"ct_dns_query\", \"DNS_QUERY_ID\"],
}

# STRICT CIC-IoT2023 mappings.
CIC_ALIASES.update({
    "packets_per_second": ["Flow Packets/s", "Flow Bytes/s"],
    "bytes_per_second": ["Flow Bytes/s"],
    "syn_ratio": ["SYN Flag Count"],
    "rst_ratio": ["RST Flag Count"],
    "avg_packet_size": ["Packet Length Mean", "Average Packet Size", "Fwd Packet Length Mean"],
    "stddev_packet_size": ["Packet Length Std", "Fwd Packet Length Std"],
    "avg_iat_us": ["Flow IAT Mean"],
    "stddev_iat_us": ["Flow IAT Std"],
})"""

if target in content:
    content = content.replace(target, replacement)
    content = content.replace("\"rate\"", "\"rate\", \"IN_PKTS\"")
    content = content.replace("\"sload\"", "\"sload\", \"IN_BYTES\"")
    content = content.replace("\"dsport\", \"ct_dst_sport_ltm\"", "\"dsport\", \"ct_dst_sport_ltm\", \"L4_DST_PORT\"")
    content = content.replace("\"sport\", \"ct_src_sport_ltm\"", "\"sport\", \"ct_src_sport_ltm\", \"L4_SRC_PORT\"")
    with open('train_ml.py', 'w', encoding='utf-8') as f:
        f.write(content)
    print("Successfully patched train_ml.py with modern dataset aliases!")
else:
    print("train_ml.py already patched or target string not found.")


In [ ]:
# @title Run ML Training
!python train_ml.py

## 4. Automatic Weight Deployment

In [ ]:
# @title Push ml_model.h to GitHub
!git config --global user.email "sentinel-bot@ai.com"
!git config --global user.name "Sentinel Cloud Trainer"
!git add decisionengine/ml_model.h
!git commit -m "chore: update ML model weights from Kaggle cloud datasets"
!git push origin main